<a href="https://colab.research.google.com/github/Aldiss1/MPPM_Aldi/blob/main/JS11/JS11-Lab1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1 — Klasifikasi Cats vs. Dogs

**Mata Kuliah:** RTI235004 — Pembelajaran Mesin (Polinema)  
**Modul:** JS11 — Convolutional Neural Network  
**Nama:** Aldi Surya Saputra


> **Cell tambahan (bukan bagian modul).**
> 1. Sejak Keras 3, `keras.preprocessing.image.ImageDataGenerator` tidak lagi diekspor,
>    sehingga baris import modul gagal. Cell pertama mengembalikan perilaku Keras 2.
> 2. Cell kedua menghubungkan Google Drive dan menyalin folder `dataset`
>    (folder yang dibagikan lewat tautan, ditambahkan sebagai shortcut di MyDrive)
>    ke `/content/dataset`.
>
> Kedua cell ini **tidak mengubah kode modul** di bawahnya.

In [18]:
import keras
from keras.src.legacy.preprocessing import image as _legacy_image
keras.preprocessing.image.ImageDataGenerator = _legacy_image.ImageDataGenerator
print('ImageDataGenerator siap dipakai:', keras.preprocessing.image.ImageDataGenerator.__name__)

ImageDataGenerator siap dipakai: ImageDataGenerator


In [3]:
# Cell tambahan (bukan bagian modul) - cek ketersediaan dataset
import os
for d in ['dataset/training_set/cats', 'dataset/training_set/dogs',
          'dataset/test_set/cats', 'dataset/test_set/dogs',
          'dataset/single_prediction']:
    print(f'{d:35s} ->', len(os.listdir(d)) if os.path.isdir(d) else 'TIDAK ADA')

dataset/training_set/cats           -> 4000
dataset/training_set/dogs           -> 4000
dataset/test_set/cats               -> 1000
dataset/test_set/dogs               -> 1000
dataset/single_prediction           -> 2


## Langkah 1 - Import Library

In [4]:
import tensorflow as tf
from keras.preprocessing.image import ImageDataGenerator

## Langkah 2 - Pra Pengolahan Data

Pada tahap ini kita akan sedikit melakukan manipulasi pada citra yang digunakan. Manipulasi yang dilakukan diantaranya adalah normalisasi nilai piksel, koreksi kemiringan, pembesaran (zoom), dan flip.

### Langkah 2.1. Pra Pengolahan Data Training

In [5]:
train_datagen = ImageDataGenerator(rescale = 1./255,
                                   shear_range = 0.2,
                                   zoom_range = 0.2,
                                   horizontal_flip = True)
training_set = train_datagen.flow_from_directory('dataset/training_set',
                                                 target_size = (64, 64),
                                                 batch_size = 32,
                                                 class_mode = 'binary')

Found 8000 images belonging to 2 classes.


### Langkah 2.2. Pra Pengolahan Data Testing

In [6]:
test_datagen = ImageDataGenerator(rescale = 1./255)
test_set = test_datagen.flow_from_directory('dataset/test_set',
                                            target_size = (64, 64),
                                            batch_size = 32,
                                            class_mode = 'binary')

Found 2000 images belonging to 2 classes.


## Langkah 3 - Pembuatan Model CNN

### Langkah 3.1.  - Inisiasi Model CNN

In [7]:
cnn = tf.keras.models.Sequential()

### Langkah 3.2. - Pembuatan Layer Konvolusi 1

In [8]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu', input_shape=[64, 64, 3]))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


### Langkah 3.3 - Pembuatan Layer Pooling 1

In [9]:
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Langkah 3.4 - Pembuatan Layer Konvolusi 2 dan Pooling 2

In [10]:
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu'))
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

### Langkah 3.5 - Flattening

In [11]:
cnn.add(tf.keras.layers.Flatten())

### Langkah 3.6 - Fully Connected Layer 1 (Input)

In [12]:
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))

### Langkah 3.7 - Fully Connected Layer 2 (Output)

In [13]:
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

### Langkah 3.8 - Compile Model CNN

In [14]:
cnn.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

Penggunaan ***loss function binary crossentropy*** dikarenakan kita hanya melakukan klasifikasi pada dua kelas, yaitu kucing dan anjing.

## Langkah 4 - Fit CNN

In [15]:
cnn.fit(x = training_set, validation_data = test_set, epochs = 25)

Epoch 1/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 71s 277ms/step - accuracy: 0.5601 - loss: 0.6837 - val_accuracy: 0.5410 - val_loss: 0.6760
Epoch 2/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 67s 267ms/step - accuracy: 0.6471 - loss: 0.6358 - val_accuracy: 0.7020 - val_loss: 0.6065
Epoch 3/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 69s 275ms/step - accuracy: 0.6914 - loss: 0.5858 - val_accuracy: 0.7000 - val_loss: 0.5756
Epoch 4/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 67s 268ms/step - accuracy: 0.7157 - loss: 0.5577 - val_accuracy: 0.7125 - val_loss: 0.5869
Epoch 5/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 68s 271ms/step - accuracy: 0.7352 - loss: 0.5228 - val_accuracy: 0.7455 - val_loss: 0.5267
Epoch 6/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 67s 269ms/step - accuracy: 0.7559 - loss: 0.4943 - val_accuracy: 0.7625 - val_loss: 0.5063
Epoch 7/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 74s 297ms/step - accuracy: 0.7699 - loss: 0.4752 - val_accuracy: 0.7660 - val_loss: 0.5004
Epoch 8/25
250/250 ━━━━━━━━━━━━━━━━━━━━ 70s 280ms/step - accuracy: 0.7843 - loss: 0

## Langkah 5 - Prediksi dengan 1 Citra

Pada langkah ini, kita akan mencoba melakukan prediksi pada 1 citra anjing dan kucing.

In [16]:
import numpy as np
from keras.preprocessing import image
test_image = image.load_img('dataset/single_prediction/cat_or_dog_1.jpg', target_size = (64, 64))
test_image = image.img_to_array(test_image)
test_image = np.expand_dims(test_image, axis = 0)
result = cnn.predict(test_image)
training_set.class_indices
if result[0][0] == 1:
  prediction = 'dog'
else:
  prediction = 'cat'

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step


> **Cell tambahan (bukan bagian modul)** — menampilkan hasil prediksi di atas.

In [17]:
# Cell tambahan (bukan bagian modul)
print('class_indices :', training_set.class_indices)
print('nilai output  :', result[0][0])
print('prediction    :', prediction)

class_indices : {'cats': 0, 'dogs': 1}
nilai output  : 1.0
prediction    : dog
